# 5.1. 层和块

块（block）可以描述单个层、由多个层组成的组件或整个模型本身。请自己完成每个代码单元。

导入所需模块：torch、torch.nn、torch.nn.functional（F）。

In [9]:
import torch
from torch import nn
from torch.nn import functional as F

用 nn.Sequential 定义一个多层感知机 net：nn.Linear(20, 256) → nn.ReLU() → nn.Linear(256, 10)；生成 X = torch.rand(2, 20) 并调用 net(X)。

In [3]:
net=nn.Sequential(
    nn.Flatten(),
    nn.Linear(20,256),
    nn.ReLU(),
    nn.Linear(256,10),
)
X=torch.randn(2,20)
net(X)


tensor([[ 0.1167, -0.2950,  0.2464,  0.2253, -0.2198, -0.2830,  0.1365,  0.2006,
         -0.2476,  0.4556],
        [-0.0746, -0.0775, -0.0584, -0.0086,  0.0017,  0.0402, -0.2390,  0.2639,
          0.4092,  0.0881]], grad_fn=<AddmmBackward0>)

### 5.1.1. 自定义块

继承 nn.Module 编写一个 MLP 类：
- __init__ 中调用 super().__init__()，声明 self.hidden = nn.Linear(20, 256)（隐藏层）、self.out = nn.Linear(256, 10)（输出层）
- forward(self, X)：return self.out(F.relu(self.hidden(X)))

实例化 net = MLP() 并用 X 调用。

In [4]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden=nn.Linear(20,2)
        self.out=nn.Linear(2,10)
    def forward(self,x):
        return self.out(F.relu(self.hidden(x)))


### 5.1.2. 顺序块

自己实现一个简化的 MySequential，与 nn.Sequential 行为一致：
- __init__(self, *args)：调用 super().__init__()，用 enumerate 遍历 args，把每个模块存入 self._modules[str(idx)]
- forward(self, X)：for block in self._modules.values(): X = block(X)，最后 return X

实例化并测试。

In [5]:

class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()
        for idx, module in enumerate(args):
            self._modules[str(idx)]=module
    
    def forward(self,X):
        for block in self._modules.values(): 
            X = block(X)
        return X





### 5.1.3. 在前向传播函数中执行代码

块可以在前向传播中做更灵活的事。实现 FixedHiddenMLP：
- __init__ 声明 self.rand_weight = torch.rand((20, 20), requires_grad=False)（不参与反向传播）和 self.linear = nn.Linear(20, 20)
- forward(self, X)：X = self.linear(X)；X = F.relu(X @ self.rand_weight + 1)；X = self.linear(X)；再用 while X.abs().sum() > 1: X /= 2 调节，最后 return X.sum()

还可把自定义块嵌进 nn.Sequential（例如精妙混合使用）。实例化并用 X 测试。

In [6]:
class FixedHiddenMLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.rand_weight=torch.rand((20,20),requires_grad=False)
        self.linear=nn.Linear(20,20)
    def forward(self,X):
        X=self.linear(X)
        X=F.relu(X@self.rand_weight+1)
        X=self.linear(X)
        while(X.abs().sum()>1):
            X/=2
        return X.sum()

net.insert(1,FixedHiddenMLP())


Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): FixedHiddenMLP(
    (linear): Linear(in_features=20, out_features=20, bias=True)
  )
  (2): Linear(in_features=20, out_features=256, bias=True)
  (3): ReLU()
  (4): Linear(in_features=256, out_features=10, bias=True)
)

### 5.1.4. 效率

（了解）框架尽力避免不必要地评估每个项，构建计算图时框架会跳过无关的项。

### 5.1.5. 小结

- 块可以组合层形成递归结构，简化复杂网络的实现
- 自定义块需要继承 nn.Module 并实现 forward 前向传播函数
- 块内部可以自由执行代码，如调用其他块、控制流、数组运算等

### 5.1.6. 练习

1. 如果 MLP 改为不加 ReLU 而是直接输出，会发生什么？
2. 尝试自定义一个由块构成的、包含若干层和参数的结构。
3. 若 forward 中用到其他 Python 控制流程（if、for 等），梯度计算是否仍正确？

In [8]:



class MLP(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.rand_weight=torch.rand((20,20),requires_grad=False)
        self.linear=nn.Linear(20,20)
    def forward(self,X):
        self.linear(X)
        self.linear(X)
        return X

M=MLP()
print(M.forward(torch.rand(20,20)))


tensor([[0.6608, 0.7258, 0.6518, 0.2796, 0.3660, 0.3703, 0.4850, 0.7820, 0.0689,
         0.7298, 0.4981, 0.8485, 0.1775, 0.1827, 0.9177, 0.9515, 0.3257, 0.4641,
         0.6995, 0.0360],
        [0.7506, 0.9250, 0.2033, 0.7086, 0.1575, 0.5990, 0.7041, 0.2755, 0.2708,
         0.2087, 0.3701, 0.4452, 0.3504, 0.7410, 0.0492, 0.4232, 0.4269, 0.1641,
         0.7646, 0.4992],
        [0.8297, 0.7816, 0.4639, 0.0141, 0.8060, 0.2413, 0.1329, 0.5543, 0.9811,
         0.0140, 0.3474, 0.2762, 0.8107, 0.5598, 0.7318, 0.7536, 0.3816, 0.2370,
         0.6890, 0.7683],
        [0.1872, 0.6490, 0.4238, 0.8221, 0.4921, 0.0431, 0.2935, 0.1285, 0.6622,
         0.5607, 0.5486, 0.5680, 0.9384, 0.7099, 0.3064, 0.9640, 0.0500, 0.4754,
         0.4215, 0.4852],
        [0.5369, 0.6833, 0.5412, 0.8442, 0.2173, 0.6293, 0.2738, 0.3040, 0.0487,
         0.8363, 0.4832, 0.3825, 0.8004, 0.9174, 0.0043, 0.6876, 0.8920, 0.2172,
         0.9748, 0.5211],
        [0.7051, 0.7548, 0.9422, 0.0769, 0.6611, 0.4885, 0.2